# species2vec (corrected)

Rebuild of the species2vec idea — learn species embeddings from spatial co-occurrence of GBIF records — with the methodology fixed.

**What was wrong in the original notebook:**

1. The whole corpus was concatenated into one space-separated string. fastText slides a context window across the entire text, so species at the end of one continent became neighbors of species at the start of the next. There was no real notion of "these species co-occur".
2. `Geohash.encode(lat, lon)` was called with default precision (12 chars ≈ millimeters), so every record had a unique cell. There were no co-occurrence bins at all — only the z-curve ordering produced pseudo-neighbors.
3. fastText was trained with character n-grams enabled. Two crocs on different continents end up similar **because their Latin names share substrings**, not because they live near each other. The model partly learned taxonomy from the labels.
4. No deduplication of occurrences, no held-out evaluation, no seed.

**This notebook:**

- Bins occurrences at a configurable geohash precision (5 chars ≈ 5 km).
- Writes one *sentence* per geohash bin — fastText's window cannot bleed across bins.
- Disables character n-grams (`minn=0, maxn=0`) so the embedding cannot cheat through the name.
- Holds out 10 % of occurrences and computes congener cosine, sympatry pair AUC, and held-out neighborhood self-rank.

In [ ]:
import pandas as pd
from pathlib import Path

from species2vec.pipeline import build_corpus, train_embeddings
from species2vec.eval import evaluate

## 1. Get data

For a quick demo we use one order via the GBIF search API. For the full-scale corpora used in the original paper, register a GBIF download (multi-GB tsv) and pass it here instead.

In [ ]:
# Shell: python -m species2vec.gbif_download_parallel --order Squamata --out data/squamata.csv --per-country 8000 --workers 6
df = pd.read_csv('data/squamata.csv')
df = df.dropna(subset=['species', 'decimalLatitude', 'decimalLongitude'])
df.shape, df['species'].nunique()

## 2. Split off a held-out set

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
mask = rng.random(len(df)) < 0.1
train_df = df[~mask].reset_index(drop=True)
hold_df  = df[mask].reset_index(drop=True)
len(train_df), len(hold_df)

## 3. Build corpus and train

`geohash_precision=4` → ~20 km cells. With ~150 k Squamata records this gives ~8 000 bins; for the full mammalia/reptilia corpora used in the original paper, `precision=5` (~5 km cells) is more appropriate.

In [ ]:
workdir = Path('runs/squamata')
workdir.mkdir(parents=True, exist_ok=True)

stats = build_corpus(
    train_df, workdir / 'corpus.txt',
    geohash_precision=5, min_bin_size=2,
)
stats

In [ ]:
train_embeddings(
    workdir / 'corpus.txt', workdir / 'squamata.vec',
    dim=100, epoch=25, window=8, min_count=3, seed=42,
)

## 4. Evaluate

Three independent checks:

- **congener_score / gap** — mean cosine to other species in the same genus minus mean cosine to random species. With character n-grams ON this gap is huge (~0.4) because the model is reading the name. With them OFF the gap reflects only spatial co-occurrence — usually small and positive (congeners tend to be sympatric).
- **sympatry_auc** — for every species pair seen in the same held-out geohash bin (label 1) versus a matched sample of pairs that never co-occur (label 0), is cosine a good discriminator? 0.5 = no signal, 1.0 = perfect.
- **neighborhood_median_rank** — for each held-out (species, bin), rank all vocab species by similarity to the bin's centroid. Median rank of the held-out species; lower is better, |V|/2 = random.

In [ ]:
hold_df = hold_df.copy()
hold_df['species'] = hold_df['species'].astype(str).str.replace(' ', '_')
report = evaluate(workdir / 'squamata.vec', hold_df, geohash_precision=4)
print(report.pretty())

## 5. Inspect neighbors

In [ ]:
from gensim.models import KeyedVectors
kv = KeyedVectors.load_word2vec_format(str(workdir / 'squamata.vec'))
kv.most_similar('Alligator_mississippiensis' if 'Alligator_mississippiensis' in kv.key_to_index else kv.index_to_key[0], topn=10)

## 6. Compare against the broken pipeline

Run `python -m species2vec.compare --csv data/squamata.csv` to train both the original-style pipeline and the corrected one on the same train/holdout split and print the eval side by side.